# M5 경제표현 + 양성 구매금액 가중학습: test-only seed 42

검증 구간을 만들지 않습니다. DAY 1~697을 모두 학습하고, 고정 100 epoch의 마지막 checkpoint를 DAY 698~704 test에서 한 번만 평가합니다. DAY 705~711은 사용하지 않습니다. 이번 노트북은 프로토콜 확인용 seed 42 한 개이며, 결과로 모형·수식·epoch·하이퍼파라미터를 다시 선택하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '215255f012907ff440388bfb190ac36d02873623'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_economic_positive_weight_test import (
    configure_m5_economic_positive_test_run,
    preflight_summary,
    run_m5_economic_positive_test,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_m5_economic_positive_test_run(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_economic_positive_weighting_test_seed42_v1',
)
summary = preflight_summary(cfg)
assert cfg.seeds == (42,)
assert summary['validation_constructed'] is False
assert summary['holdout_evaluation'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m5_economic_positive_test(cfg)

In [ ]:
from IPython.display import display

print('1) seed 42 test 절대지표')
display(result_df)
print('2) seed 평균표 — 현재는 seed 42 한 개')
display(result_df.attrs['mean'])
print('3) 동일 seed 대조군 비교')
display(result_df.attrs['comparison'])
print('4) 동일 seed 평균 차이 — 현재는 seed 42 한 개')
display(result_df.attrs['paired_mean'])
print("5) M2 × M4' 상호작용")
display(result_df.attrs['interaction'])
print('6) 사전 고정 기준의 기술적 판독 — 모형선택에 사용하지 않음')
print(json.dumps(result_df.attrs['descriptive_reading'], ensure_ascii=False, indent=2))
print('7) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))